In [ ]:
import pandas as pd
import numpy as np
import duckdb

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pd.set_option('display.max_colwidth',None) #permet d'avoir le contenu complet des colonnes dans le dataframe

Clean transactions_train.csv

In [ ]:
# Création d'une connexion DuckDB
con = duckdb.connect()

# Définition de la requête SQL pour filtrer directement le fichier transactions_train.csv avec l'année 2019
# DuckDB permet de requêter un fichier .csv comme une table
query = """
SELECT * FROM '/content/drive/MyDrive/Projet 3/dataset/transactions_train.csv'
WHERE t_dat >= '2019-01-01' AND t_dat <= '2019-12-31'
"""

print("Début de l'extraction des données 2019...")

# Exécution et conversion directe en DataFrame Pandas
df_2019 = con.execute(query).df()

print(f"Extraction terminée. Nombre de lignes récupérées : {len(df_2019)}")

# Vérification des premières lignes
df_2019.head()


Renommer la colonne t_dat et sales_channel_id

In [ ]:
df_2019 = df_2019.rename(columns={
    "t_dat": "date",
    "sales_channel_id": "type_commerce"
})


Grouper les commandes en double et creer une colonne quantité

In [ ]:
df_2019 = (
    df_2019
    .groupby(
        ["date", "customer_id", "article_id", "price", "type_commerce"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "quantity"}))

Selectionner seulement les commandes de 2019

In [ ]:
df_2019 = df_2019[df_2019["date"].dt.year == 2019]

Creer la colonne CA

In [ ]:
df_2019["revenue"] = df_2019["price"] * df_2019["quantity"]

In [ ]:
ca_total = df_2019["revenue"].sum()
print(f"CA total 2019 : {ca_total:,.2f}")


CA total 2019 : 455,604.88


In [ ]:
df_2019.to_csv("transactions_clean.csv", index=False)